# Prokka → ESM3 → DALI Workflow

这个工作流程将：
1. 接受 FNA（核酸序列）文件作为输入
2. 使用 Prokka 进行基因注释和蛋白质预测
3. 将 Prokka 输出的蛋白质序列逐条输入 ESM3 进行结构预测
4. 生成符合 DALI 输入标准的 PDB 文件
5. 保存所有结果到服务器目录

## 系统要求
- JupyterLab/JupyterHub 服务器环境（推荐使用 GPU）
- 约 10-20 GB 磁盘空间
- 运行时间取决于序列数量和长度

## 自动安装功能 🆕
- **无需预装 conda/mamba**：Notebook 会自动安装 micromamba
- **自动创建环境**：自动安装 Prokka 及其依赖
- **一键运行**：上传 Notebook 即可在全新服务器上运行

如需禁用自动安装，设置环境变量：`PROTFLOW_AUTO_INSTALL_MICROMAMBA=0`

## 1. 环境检测与设置

In [ ]:
import sys
import os
import subprocess
import shutil
from pathlib import Path
def _str_to_bool(value, default=False):
    if value is None:
        return default
    normalized = str(value).strip().lower()
    if normalized == '':
        return default
    return normalized in {"1", "true", "yes", "y"}
def _load_env_file(env_path: Path):
    if not env_path.exists():
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, val = line.split('=', 1)
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))
PROJECT_ROOT = Path(os.environ.get('PROTFLOW_ROOT', Path.cwd())).resolve()
ENV_FILE = Path(os.environ.get('PROTFLOW_ENV_FILE', PROJECT_ROOT / '.env'))
_load_env_file(ENV_FILE)
IN_COLAB = 'google.colab' in sys.modules
IN_JUPYTERHUB = bool(os.environ.get('JUPYTERHUB_SERVICE_PREFIX'))
if IN_COLAB:
    print("✓ 运行在 Google Colab")
    from google.colab import files, drive  # type: ignore
elif IN_JUPYTERHUB:
    print("✓ 运行在 JupyterHub/JupyterLab 服务器环境")
else:
    print("✓ 运行在本地环境")
PROKKA_ENV_NAME = os.environ.get('PROKKA_ENV_NAME', 'prokka')
AUTO_CREATE_PROKKA_ENV = _str_to_bool(os.environ.get('PROTFLOW_AUTO_CREATE_PROKKA', '1'), default=True)
if IN_COLAB:
    WORK_DIR = Path('/content/prokka_esm3_workflow')
else:
    default_runs_dir = PROJECT_ROOT / 'prokka_esm3_runs'
    WORK_DIR = Path(os.environ.get('PROTFLOW_WORKDIR', default_runs_dir)).expanduser()
WORK_DIR.mkdir(exist_ok=True, parents=True)
if IN_COLAB:
    os.chdir(WORK_DIR)
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU 可用: {torch.cuda.get_device_name(0)}")
        print(f"  显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        print("⚠ 未检测到 GPU，建议在运行时设置中启用 GPU")
else:
    print(f"项目根目录: {PROJECT_ROOT}")
    print(f"输出目录: {WORK_DIR.resolve()}")
    if IN_JUPYTERHUB:
        print("提示: 当前处于远程 JupyterLab/Hub，会话结束后输出仍保存在服务器上。")
    else:
        print("提示: 输出内容将写入上述目录，可用终端或文件浏览器访问。")
print(f"使用 PROKKA_ENV_NAME={PROKKA_ENV_NAME}")
print(f"AUTO_CREATE_PROKKA_ENV={'ON' if AUTO_CREATE_PROKKA_ENV else 'OFF'}")
print(f"\n工作目录: {WORK_DIR.resolve()}")


## 2. 安装依赖

本 Notebook 支持自动安装所需的所有依赖：
- **Micromamba**：如果系统中没有 conda/mamba，将自动下载并安装到 `~/.local/bin/`
- **Prokka**：自动创建 conda 环境并安装 Prokka 及其依赖

整个过程自动化，无需手动操作。首次运行约需 5-10 分钟。

In [ ]:
import subprocess
import shutil
import platform
import tempfile
from pathlib import Path

def install_micromamba():
    """
    自动安装 micromamba 到用户目录
    支持 Linux 和 macOS
    """
    print("="*60)
    print("自动安装 Micromamba")
    print("="*60)

    system = platform.system()
    machine = platform.machine()

    # 确定安装目录
    install_dir = Path.home() / '.local' / 'bin'
    install_dir.mkdir(parents=True, exist_ok=True)
    micromamba_bin = install_dir / 'micromamba'

    # 检查是否已经安装
    if micromamba_bin.exists():
        print(f"✅ Micromamba 已存在: {micromamba_bin}")
        # 添加到 PATH
        if str(install_dir) not in os.environ.get('PATH', ''):
            os.environ['PATH'] = f"{install_dir}:{os.environ.get('PATH', '')}"
        return str(micromamba_bin)

    print(f"\n检测到系统: {system} ({machine})")
    print(f"安装目录: {install_dir}")

    # 确定下载 URL
    if system == 'Linux':
        if machine == 'x86_64':
            url = 'https://micro.mamba.pm/api/micromamba/linux-64/latest'
        elif machine == 'aarch64':
            url = 'https://micro.mamba.pm/api/micromamba/linux-aarch64/latest'
        else:
            raise RuntimeError(f"不支持的 Linux 架构: {machine}")
    elif system == 'Darwin':  # macOS
        if machine == 'arm64':
            url = 'https://micro.mamba.pm/api/micromamba/osx-arm64/latest'
        else:
            url = 'https://micro.mamba.pm/api/micromamba/osx-64/latest'
    else:
        raise RuntimeError(f"不支持的操作系统: {system}")

    print(f"\n📥 正在下载 micromamba...")
    print(f"   URL: {url}")

    try:
        # 下载并解压
        import tarfile
        import urllib.request

        with tempfile.NamedTemporaryFile(suffix='.tar.bz2', delete=False) as tmp_file:
            tmp_path = Path(tmp_file.name)

            # 下载文件
            urllib.request.urlretrieve(url, tmp_path)
            print(f"✅ 下载完成: {tmp_path.stat().st_size / 1024 / 1024:.2f} MB")

            # 解压
            print("📦 正在解压...")
            with tarfile.open(tmp_path, 'r:bz2') as tar:
                # 提取 bin/micromamba
                for member in tar.getmembers():
                    if member.name.endswith('bin/micromamba') or member.name == 'bin/micromamba':
                        member.name = 'micromamba'  # 重命名为 micromamba
                        tar.extract(member, install_dir)
                        break

            # 删除临时文件
            tmp_path.unlink()

        # 设置执行权限
        micromamba_bin.chmod(0o755)

        print(f"✅ Micromamba 安装成功: {micromamba_bin}")

        # 添加到 PATH
        os.environ['PATH'] = f"{install_dir}:{os.environ.get('PATH', '')}"

        # 初始化 micromamba
        print("\n🔧 正在初始化 micromamba...")
        try:
            subprocess.run([str(micromamba_bin), 'shell', 'init', '-s', 'bash', '-p', str(Path.home() / 'micromamba')],
                          capture_output=True, check=False)
            print("✅ Micromamba 初始化完成")
        except Exception as e:
            print(f"⚠️ 初始化警告（可忽略）: {e}")

        return str(micromamba_bin)

    except Exception as e:
        print(f"\n❌ 安装失败: {e}")
        print("\n请手动安装 micromamba:")
        print(f"  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
        print(f"  macOS: brew install micromamba")
        raise

# 检测可用的 conda/mamba 包管理器
CONDA_BIN = None
AUTO_INSTALLED_MICROMAMBA = False

# 首先检查 micromamba 和 mamba (更快)
for cmd in ['micromamba', 'mamba']:
    bin_path = shutil.which(cmd)
    if bin_path:
        CONDA_BIN = (cmd, bin_path)
        print(f"✅ 检测到 {cmd}: {bin_path}")
        break

# 如果没有，检查 conda
if not CONDA_BIN:
    # conda 通常是 shell 函数，需要特殊处理
    conda_exe = os.environ.get('CONDA_EXE') or shutil.which('conda')
    if conda_exe:
        CONDA_BIN = ('conda', conda_exe)
        print(f"✅ 检测到 conda: {conda_exe}")

# 如果都没有，尝试自动安装 micromamba
if not CONDA_BIN:
    print("\n⚠️ 未检测到 conda/mamba/micromamba")

    # 检查是否允许自动安装
    auto_install = os.environ.get('PROTFLOW_AUTO_INSTALL_MICROMAMBA', '1')
    if auto_install in ('1', 'true', 'True', 'yes', 'YES'):
        try:
            print("\n🚀 正在自动安装 micromamba...")
            print("   (如不需要，请设置环境变量: PROTFLOW_AUTO_INSTALL_MICROMAMBA=0)")

            micromamba_path = install_micromamba()
            CONDA_BIN = ('micromamba', micromamba_path)
            AUTO_INSTALLED_MICROMAMBA = True

            print(f"\n✅ Micromamba 安装并配置成功！")
            print(f"   路径: {micromamba_path}")
            print(f"   (已添加到当前会话的 PATH)")

        except Exception as e:
            print(f"\n❌ 自动安装失败: {e}")
            print("\n请选择以下任一方式手动安装:")
            print("\n方式 1 - 安装 micromamba (推荐):")
            print("  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
            print("  macOS: brew install micromamba")
            print("\n方式 2 - 安装 conda/mamba:")
            print("  https://docs.conda.io/en/latest/miniconda.html")
            print("\n方式 3 - 设置已安装的 Prokka:")
            print("  export PROKKA_BIN=/path/to/prokka")
            print("="*60)
    else:
        print("\n自动安装已禁用（PROTFLOW_AUTO_INSTALL_MICROMAMBA=0）")
        print("\n请选择以下任一方式手动安装:")
        print("\n方式 1 - 安装 micromamba (推荐):")
        print("  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
        print("  macOS: brew install micromamba")
        print("\n方式 2 - 安装 conda/mamba:")
        print("  https://docs.conda.io/en/latest/miniconda.html")
        print("="*60)

def ensure_prokka_available(env_name: str = 'prokka', auto_create: bool = True):
    """检查并确保 Prokka 可用"""
    # 1. 检查环境变量 PROKKA_BIN
    override = os.environ.get('PROKKA_BIN')
    if override:
        prokka_path = Path(override).expanduser()
        if not prokka_path.exists():
            raise FileNotFoundError(f"PROKKA_BIN 指向的文件不存在: {prokka_path}")
        print(f"✅ 使用 PROKKA_BIN: {prokka_path}")
        return [str(prokka_path)]

    # 2. 检查是否有 conda/mamba 环境
    if CONDA_BIN:
        conda_cmd, conda_path = CONDA_BIN

        # 检查环境是否存在
        try:
            env_list = subprocess.run(
                [conda_path, 'env', 'list'],
                capture_output=True,
                text=True,
                check=True
            )
            env_exists = env_name in env_list.stdout
        except Exception:
            env_exists = False

        # 构建 prokka 命令
        if conda_cmd == 'conda':
            # conda 需要先激活环境，这里使用 conda run
            prokka_cmd = ['conda', 'run', '-n', env_name, 'prokka']
        else:
            # mamba/micromamba 使用 run
            prokka_cmd = [conda_path, 'run', '-n', env_name, 'prokka']

        # 如果环境不存在，尝试创建
        if not env_exists:
            if not auto_create:
                raise RuntimeError(
                    f"未检测到 {conda_cmd} 环境 '{env_name}'，并且自动创建被禁用。\n"
                    f"请手动创建: {conda_cmd} create -n {env_name} -c conda-forge -c bioconda prokka"
                )

            print(f"📦 正在使用 {conda_cmd} 创建环境: {env_name}")
            print("   这可能需要几分钟时间...")

            try:
                # 创建环境并安装 prokka
                create_cmd = [
                    conda_path, 'create', '-y', '-n', env_name,
                    '-c', 'conda-forge', '-c', 'bioconda', '-c', 'defaults',
                    'prokka'
                ]
                if conda_cmd != 'conda':
                    create_cmd.insert(2, 'python=3.12')  # mamba/micromamba 可以指定 python 版本

                subprocess.run(create_cmd, check=True)

                print('📥 正在初始化 Prokka 数据库...')
                subprocess.run(prokka_cmd + ['--setupdb'], check=True)
                print('✅ Prokka 环境创建成功！')

            except subprocess.CalledProcessError as e:
                raise RuntimeError(
                    f"创建 Prokka 环境失败。\n"
                    f"请手动安装: {conda_cmd} create -n {env_name} -c conda-forge -c bioconda prokka"
                ) from e

        return prokka_cmd

    # 3. 检查系统路径中的 prokka
    prokka_cli = shutil.which('prokka')
    if prokka_cli:
        print(f"✅ 使用系统中的 Prokka: {prokka_cli}")
        return [prokka_cli]

    # 4. 都没找到，抛出错误
    raise RuntimeError(
        '未检测到 Prokka。请选择以下任一方式:\n'
        '1. 安装 conda/mamba 并重新运行此单元格（将自动创建环境）\n'
        '2. 手动安装 Prokka 后设置环境变量: export PROKKA_BIN=/path/to/prokka\n'
        '3. 在已激活的 conda 环境中安装: conda install -c bioconda prokka'
    )
try:
    PROKKA_CMD = ensure_prokka_available(PROKKA_ENV_NAME, AUTO_CREATE_PROKKA_ENV)
    version_info = subprocess.run(
        PROKKA_CMD + ['--version'],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()
    print(f"\n✅ Prokka 就绪: {version_info}")
    print(f"   调用命令: {' '.join(PROKKA_CMD)}")
except Exception as exc:
    print(f"\n❌ Prokka 配置失败: {exc}")
    print('请根据上方提示在服务器上准备 Prokka 再继续运行剩余单元格。')
    raise


In [ ]:
# Prokka 已在上一步安装，此单元格已合并
pass

In [ ]:
import sys
INSTALL_PYTHON_DEPS = _str_to_bool(os.environ.get('PROTFLOW_AUTO_INSTALL_PY', '0'))
requirements_file = PROJECT_ROOT / 'requirements.txt'
if INSTALL_PYTHON_DEPS:
    if requirements_file.exists():
        print(f"📦 正在安装 Python 依赖: {requirements_file}")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], check=True)
    else:
        print('⚠️ 未找到 requirements.txt，跳过自动安装')
else:
    print('Python 依赖请在终端中预先安装（一次即可）：')
    print(f"  pip install -r {requirements_file}")
    print('如需在 notebook 中自动安装，可设置环境变量 PROTFLOW_AUTO_INSTALL_PY=1')


## 3. 安装 Python 包依赖

安装工作流所需的 Python 包（ESM3, BioPython, PyTorch 等）

In [ ]:
import subprocess
import sys
from pathlib import Path

print("="*60)
print("安装 Python 包依赖")
print("="*60)

# 检查是否有 requirements.txt
requirements_file = PROJECT_ROOT / 'requirements.txt'

if requirements_file.exists():
    print(f"\n✅ 发现 requirements.txt: {requirements_file}")
    print("\n📦 正在安装依赖包...")
    print("   这可能需要几分钟时间...\n")
    
    try:
        # 使用 pip 安装依赖
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)],
            check=True,
            capture_output=True,
            text=True
        )
        print("✅ 依赖包安装成功！")
        
        # 显示已安装的关键包
        print("\n已安装的关键包:")
        key_packages = ['esm', 'torch', 'biopython', 'huggingface_hub', 'tqdm', 'pandas']
        for pkg in key_packages:
            try:
                result = subprocess.run(
                    [sys.executable, '-m', 'pip', 'show', pkg],
                    capture_output=True,
                    text=True,
                    check=False
                )
                if result.returncode == 0:
                    for line in result.stdout.split('\n'):
                        if line.startswith('Version:'):
                            version = line.split(':')[1].strip()
                            print(f"  ✓ {pkg}: {version}")
                            break
            except:
                pass
                
    except subprocess.CalledProcessError as e:
        print(f"❌ 安装失败: {e}")
        print(f"\n错误输出:\n{e.stderr}")
        print("\n请手动安装:")
        print(f"  pip install -r {requirements_file}")
        raise
else:
    print(f"\n⚠️ 未找到 requirements.txt: {requirements_file}")
    print("\n手动安装核心依赖:")
    
    core_packages = [
        'esm>=3.2.1',
        'huggingface_hub>=1.0.0',
        'biopython>=1.85',
        'pandas>=2.3.3',
        'numpy>=2.0.0',
        'tqdm>=4.67.1',
        'torch>=2.0.0',
    ]
    
    print("\n📦 正在安装核心包...")
    for pkg in core_packages:
        print(f"   安装 {pkg}...")
        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', pkg],
                check=True,
                capture_output=True,
                text=True
            )
            print(f"   ✓ {pkg}")
        except subprocess.CalledProcessError as e:
            print(f"   ✗ {pkg} 安装失败")
            print(f"   错误: {e.stderr}")
    
    print("\n✅ 核心包安装完成")

print("\n" + "="*60)
print("Python 环境准备完成")
print("="*60)

## 4. HuggingFace 认证

**重要**：ESM3 模型需要 HuggingFace 认证

### 准备工作（首次使用）:

1. 访问 [ESM3 模型页面](https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1)，点击 **"Request access"** 并同意条款（自动批准）
2. 访问 [HuggingFace Tokens](https://huggingface.co/settings/tokens)，创建一个具有 **Read** 权限的 token
3. 在下面的单元格中输入 token

**安全提示**：Token 是私密的，Colab 会话结束后自动清除

In [ ]:
from huggingface_hub import login
import os
print('=' * 60)
print('🔑 Hugging Face Authentication Required')
print('=' * 60)
print('To use ESM3 model, you need a Hugging Face token.')
print('1. Go to: https://huggingface.co/settings/tokens')
print('2. Create a token with READ access')
print('3. Grant access to: EvolutionaryScale/esm3-sm-open-v1')
print('4. Paste the token below (or set HF_TOKEN / HUGGINGFACE_TOKEN env variable)')
print(f"   可在 {ENV_FILE} 中写入 HF_TOKEN=xxx 以便 Notebook 自动读取")
print('=' * 60)
HF_TOKEN = os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACE_TOKEN')
try:
    if HF_TOKEN:
        print('Using token from environment variable...')
        login(token=HF_TOKEN)
        print('✅ Logged in successfully!')
    else:
        login()  # interactive prompt on Colab / local
        print('✅ Logged in successfully!')
except Exception as e:
    print(f'❌ Login failed: {e}')
    print('Please check your token and try again.')
    raise


## 5. 导入必要的库

In [ ]:
import subprocess
import shutil
from datetime import datetime
from typing import List, Optional

import torch
from Bio import SeqIO
from tqdm.auto import tqdm

print("✓ 所有库导入成功")

## 6. 定义工作流函数

In [ ]:
class ProkkaESM3Pipeline:
    """
    Prokka -> ESM3 -> DALI 工作流管道
    """
    
    def __init__(self, work_dir: Path, prokka_cmd: Optional[List[str]] = None):
        self.work_dir = Path(work_dir)
        self.prokka_dir = self.work_dir / "prokka_output"
        self.pdb_dir = self.work_dir / "esm3_structures"
        self.dali_dir = self.work_dir / "dali_ready"
        
        # 创建目录
        for d in [self.prokka_dir, self.pdb_dir, self.dali_dir]:
            d.mkdir(exist_ok=True, parents=True)
        
        self.model = None
        self.device = None
        self.prokka_cmd = prokka_cmd or PROKKA_CMD
        if not self.prokka_cmd:
            raise RuntimeError('Prokka 未在环境中配置，请先运行依赖安装步骤。')
    
    def run_prokka(self, 
                   fna_file: Path, 
                   prefix: str = "sample",
                   kingdom: str = "Bacteria",
                   cpus: int = 2,
                   **kwargs) -> Path:
        """
        运行 Prokka 进行基因注释
        
        Args:
            fna_file: 输入的 FNA 文件路径
            prefix: 输出文件前缀
            kingdom: 生物界（Bacteria, Archaea, Viruses）
            cpus: 使用的 CPU 核心数
            **kwargs: 其他 Prokka 参数
        
        Returns:
            Prokka 输出目录路径
        """
        print(f"\n{'='*60}")
        print("步骤 1: 运行 Prokka 进行基因注释")
        print(f"{'='*60}")
        
        output_dir = self.prokka_dir / prefix
        
        # 构建 Prokka 命令 - 使用 micromamba run
        cmd = [
            "micromamba", "run", "-n", "prokka", "prokka",
            "--outdir", str(output_dir),
            "--prefix", prefix,
            "--kingdom", kingdom,
            "--cpus", str(cpus),
            "--force",  # 覆盖已存在的输出
        ]
        
        # 添加额外参数
        for key, value in kwargs.items():
            cmd.extend([f"--{key}", str(value)])
        
        cmd.append(str(fna_file))
        
        print(f"运行命令: {' '.join(cmd)}")
        print("\n正在运行 Prokka（这可能需要几分钟）...")
        
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print("\n✓ Prokka 运行成功！")
            
            # 显示统计信息
            stats_file = output_dir / f"{prefix}.txt"
            if stats_file.exists():
                print("\n注释统计:")
                print(stats_file.read_text())
            
            return output_dir
            
        except subprocess.CalledProcessError as e:
            print(f"\n✗ Prokka 运行失败: {e}")
            print(f"错误输出: {e.stderr}")
            raise
    
    def load_esm3_model(self, model_name: str = 'esm3-sm-open-v1'):
        """
        加载 ESM3 模型（参照 ProtFlow.ipynb 的实现）

        Args:
            model_name: ESM3 模型名称
        """
        print(f"\n{'='*60}")
        print("步骤 2: 加载 ESM3 模型")
        print(f"{'='*60}")
        
        # 自动检测设备
        if torch.cuda.is_available():
            self.device = 'cuda'
            print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
            print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        else:
            self.device = 'cpu'
            print("⚠️ No GPU detected. Model will run on CPU (slower).")
            print("   Tip: In Colab, enable GPU via Runtime → Change runtime type → GPU")

        print()
        print(f"Loading ESM3-sm model (this may take a few minutes)...")

        try:
            from esm.models.esm3 import ESM3

            # from_pretrained 会自动使用已登录的 HuggingFace session
            self.model = ESM3.from_pretrained(model_name).to(self.device)
            self.model.eval()

            print(f"✅ Model loaded successfully on {self.device}")
            print()
        except Exception as e:
            print(f"❌ Failed to load model: {e}")
            print("\nCommon issues:")
            print("  - Insufficient GPU memory (try restarting runtime)")
            print("  - Network timeout (try running the cell again)")
            print("  - Missing HuggingFace token (re-run Cell 3)")
            print("  - Missing ESM3 access (visit https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1)")
            raise

    def predict_structures(self,
                          prokka_dir: Path,
                          prefix: str,
                          num_steps: int = 8,
                          max_length: int = 400,
                          min_length: int = 30) -> List[Path]:
        """
        使用 ESM3 预测蛋白质结构
        
        Args:
            prokka_dir: Prokka 输出目录
            prefix: Prokka 输出前缀
            num_steps: ESM3 生成步数
            max_length: 最大序列长度（过长的序列会被跳过）
            min_length: 最小序列长度
        
        Returns:
            生成的 PDB 文件路径列表
        """
        print(f"\n{'='*60}")
        print("步骤 3: 使用 ESM3 预测蛋白质结构")
        print(f"{'='*60}")
        
        if self.model is None:
            self.load_esm3_model()
        
        from esm.sdk.api import ESMProtein, GenerationConfig
        
        # 读取 Prokka 输出的蛋白质序列
        faa_file = prokka_dir / f"{prefix}.faa"
        
        if not faa_file.exists():
            raise FileNotFoundError(f"找不到 Prokka 蛋白质文件: {faa_file}")
        
        # 解析序列
        sequences = list(SeqIO.parse(faa_file, "fasta"))
        print(f"\n从 Prokka 读取到 {len(sequences)} 条蛋白质序列")
        
        # 过滤序列
        filtered_seqs = [
            seq for seq in sequences 
            if min_length <= len(seq.seq) <= max_length
        ]
        
        skipped = len(sequences) - len(filtered_seqs)
        if skipped > 0:
            print(f"跳过 {skipped} 条序列（长度不在 {min_length}-{max_length} 范围内）")
        
        print(f"将预测 {len(filtered_seqs)} 条序列的结构\n")
        
        pdb_files = []
        success_count = 0
        error_count = 0
        
        # 逐条预测
        for rec in tqdm(filtered_seqs, desc="预测结构"):
            try:
                seq = str(rec.seq)
                # 清理 ID 以用作文件名
                name = rec.id.replace('|', '_').replace('/', '_').replace('\\', '_')[:100]
                pdb_file = self.pdb_dir / f"{name}.pdb"
                
                # 跳过已存在的文件
                if pdb_file.exists():
                    pdb_files.append(pdb_file)
                    success_count += 1
                    continue
                
                # 使用 ESM3 生成结构
                protein = ESMProtein(sequence=seq)
                protein = self.model.generate(
                    protein, 
                    GenerationConfig(track='structure', num_steps=num_steps)
                )
                
                # 保存为 PDB
                protein.to_pdb(str(pdb_file))
                pdb_files.append(pdb_file)
                success_count += 1
                
            except Exception as e:
                print(f"\n预测失败 {rec.id}: {e}")
                error_count += 1
                continue
        
        print(f"\n✓ 结构预测完成！")
        print(f"  成功: {success_count}")
        print(f"  失败: {error_count}")
        
        return pdb_files
    
    def _generate_pdb_id(self, used_ids: set) -> str:
        """
        生成符合 PDB 标准的 4 字符 ID (数字和大写字母)

        Args:
            used_ids: 已使用的 ID 集合

        Returns:
            新的唯一 PDB ID
        """
        import random
        import string

        chars = string.digits + string.ascii_uppercase
        while True:
            new_id = ''.join(random.choices(chars, k=4))
            if new_id not in used_ids:
                used_ids.add(new_id)
                return new_id

    def prepare_for_dali(self, pdb_files: List[Path]) -> Path:
        """
        准备符合 DALI 输入标准的文件
        
        DALI 要求：
        - 文件名格式：pdb<4字符ID>.ent
        - 4字符ID由数字和大写字母组成
        - 生成映射文件以追踪原始文件名

        Args:
            pdb_files: PDB 文件路径列表
        
        Returns:
            DALI 输出目录路径
        """
        print(f"\n{'='*60}")
        print("步骤 4: 准备 DALI 输入文件")
        print(f"{'='*60}")
        print("\nDALI 要求结构文件命名符合 PDB 数据库规范")
        print("正在转换文件名为：pdb<4字符ID>.ent 格式...\n")

        used_ids = set()
        mapping = []  # (dali_name, original_name) 对

        # 转换文件名并复制到 DALI 目录
        for pdb_file in tqdm(pdb_files, desc="转换并复制文件"):
            # 生成唯一的 PDB ID
            pdb_id = self._generate_pdb_id(used_ids)
            dali_name = f"pdb{pdb_id}.ent"
            dest = self.dali_dir / dali_name

            # 复制文件
            if not dest.exists():
                shutil.copy2(pdb_file, dest)

            # 记录映射
            mapping.append((dali_name, pdb_file.name))

        # 创建文件列表
        file_list = self.dali_dir / "pdb_list.txt"
        with open(file_list, 'w') as f:
            for dali_name, _ in mapping:
                f.write(f"{dali_name}\n")

        # 创建映射文件
        mapping_file = self.dali_dir / "pdb_id_mapping.tsv"
        with open(mapping_file, 'w') as f:
            f.write("DALI_Name\tOriginal_Name\n")
            for dali_name, orig_name in mapping:
                f.write(f"{dali_name}\t{orig_name}\n")

        # 创建 README
        readme = self.dali_dir / "README.txt"
        with open(readme, 'w') as f:
            f.write("DALI 输入文件\n")
            f.write("="*60 + "\n\n")
            f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"PDB 文件数量: {len(pdb_files)}\n\n")
            f.write("文件说明:\n")
            f.write("  - pdb*.ent: 符合 DALI 命名标准的结构文件\n")
            f.write("  - pdb_list.txt: DALI 格式文件名列表\n")
            f.write("  - pdb_id_mapping.tsv: DALI 名称与原始名称的映射表\n\n")
            f.write("使用说明:\n")
            f.write("1. 所有 .ent 文件已转换为 DALI 兼容格式 (pdb<4字符>.ent)\n")
            f.write("2. 使用 pdb_id_mapping.tsv 查找原始蛋白质名称\n")
            f.write("3. 访问 DALI 服务器: http://ekhidna2.biocenter.helsinki.fi/dali/\n")
            f.write("4. 上传 .ent 文件进行结构比对分析\n")
            f.write("5. pdb_list.txt 包含所有可用于 DALI 的文件列表\n")

        print(f"\n✓ DALI 文件准备完成！")
        print(f"  位置: {self.dali_dir}")
        print(f"  文件数: {len(pdb_files)}")
        print(f"  文件名格式: pdb<4字符ID>.ent")
        print(f"  映射文件: {mapping_file.name}")

        return self.dali_dir
    
    def run_full_pipeline(self,
                         fna_file: Path,
                         prefix: str = "sample",
                         kingdom: str = "Bacteria",
                         num_steps: int = 8,
                         max_seq_length: int = 400,
                         **prokka_kwargs) -> Path:
        """
        运行完整工作流
        
        Args:
            fna_file: 输入的 FNA 文件
            prefix: 输出文件前缀
            kingdom: 生物界
            num_steps: ESM3 生成步数
            max_seq_length: 最大序列长度
            **prokka_kwargs: Prokka 额外参数
        
        Returns:
            DALI 输出目录路径
        """
        start_time = datetime.now()
        print(f"\n{'='*60}")
        print("Prokka → ESM3 → DALI 工作流")
        print(f"{'='*60}")
        print(f"开始时间: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"输入文件: {fna_file}")
        print(f"输出前缀: {prefix}")
        
        # 步骤 1: Prokka 注释
        prokka_dir = self.run_prokka(fna_file, prefix, kingdom, **prokka_kwargs)
        
        # 步骤 2-3: ESM3 结构预测
        pdb_files = self.predict_structures(
            prokka_dir, 
            prefix, 
            num_steps=num_steps,
            max_length=max_seq_length
        )
        
        # 步骤 4: 准备 DALI 文件
        dali_dir = self.prepare_for_dali(pdb_files)

        end_time = datetime.now()
        duration = end_time - start_time
        
        print(f"\n{'='*60}")
        print("✓ 工作流完成！")
        print(f"{'='*60}")
        print(f"总耗时: {duration}")
        print(f"\n结果保存在: {self.work_dir}")
        print(f"  - Prokka 注释: {self.prokka_dir}")
        print(f"  - ESM3 结构: {self.pdb_dir}")
        print(f"  - DALI 文件: {dali_dir}")

        return dali_dir

print("✓ 工作流类定义完成")

## 7. 上传输入文件

上传你的 FNA（核酸序列）文件

In [ ]:
from pathlib import Path
DEFAULT_INPUT = os.environ.get('PROTFLOW_INPUT_FNA')
if not DEFAULT_INPUT:
    candidate = PROJECT_ROOT / 'example_input.fna'
    if candidate.exists():
        DEFAULT_INPUT = str(candidate)
if IN_COLAB:
    print('请上传 FNA 文件...')
    uploaded = files.upload()
    fna_files = [f for f in uploaded.keys() if f.endswith(('.fna', '.fa', '.fasta'))]
    if not fna_files:
        raise ValueError('未找到 FNA 文件，请确保上传的是 .fna/.fa/.fasta 格式的文件')
    input_fna = Path(fna_files[0])
    print(f"\n✓ 上传成功: {input_fna}")
    print(f"  文件大小: {input_fna.stat().st_size / 1024:.2f} KB")
else:
    try:
        import ipywidgets as widgets
        from IPython.display import display
        HAS_WIDGETS = True
    except Exception:
        HAS_WIDGETS = False
    input_fna = None
    def _set_input(path_str: str):
        path_obj = Path(path_str).expanduser()
        if path_obj.exists():
            globals()['input_fna'] = path_obj
            return True, path_obj
        return False, path_obj
    if HAS_WIDGETS:
        path_widget = widgets.Text(
            value=DEFAULT_INPUT or '',
            description='FNA 路径:',
            placeholder='/path/to/genome.fna',
            layout=widgets.Layout(width='80%')
        )
        status = widgets.HTML()
        def _refresh(change=None):
            ok, candidate = _set_input(path_widget.value)
            if ok:
                status.value = f"<span style='color:green'>✓ 找到文件: {candidate}</span>"
            else:
                status.value = f"<span style='color:red'>✗ 找不到文件: {candidate}</span>"
        path_widget.observe(_refresh, names='value')
        _refresh()
        display(widgets.VBox([path_widget, status]))
        if 'input_fna' in globals():
            print(f'当前输入文件: {input_fna}')
        else:
            print('请在文本框中输入服务器上的 FNA 文件路径。')
    else:
        if DEFAULT_INPUT:
            ok, candidate = _set_input(DEFAULT_INPUT)
            if ok:
                print(f'✓ 输入文件: {candidate}')
            else:
                print(f'⚠️ 默认路径不存在: {candidate}')
        else:
            print('请设置 PROTFLOW_INPUT_FNA 环境变量或手动将 input_fna 设为本地 FNA 文件路径。')


## 8. 配置工作流参数

In [ ]:
# 基本参数
OUTPUT_PREFIX = "my_genome"  # 输出文件前缀
KINGDOM = "Bacteria"  # 生物界: Bacteria, Archaea, 或 Viruses

# Prokka 参数
GENUS = None  # 例如: "Escherichia" (可选)
SPECIES = None  # 例如: "coli" (可选)
STRAIN = None  # 例如: "K12" (可选)

# ESM3 参数
NUM_STEPS = 8  # 生成步数 (8-16，越大越慢但质量可能更好)
MAX_SEQ_LENGTH = 400  # 最大序列长度（超过此长度的序列会被跳过）
MIN_SEQ_LENGTH = 30  # 最小序列长度

# CPU 核心数
CPUS = 2

print("配置参数:")
print(f"  输出前缀: {OUTPUT_PREFIX}")
print(f"  生物界: {KINGDOM}")
print(f"  ESM3 步数: {NUM_STEPS}")
print(f"  序列长度范围: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH}")

## 9. 运行 Prokka 基因注释

首先运行 Prokka 进行基因注释

In [ ]:
# 创建管道实例
pipeline = ProkkaESM3Pipeline(WORK_DIR)
if 'input_fna' not in globals():
    raise ValueError('请先在第 6 步设置 input_fna 路径。')
if not Path(input_fna).exists():
    raise FileNotFoundError(f'输入文件不存在: {input_fna}')
# 准备 Prokka 参数
prokka_params = {
    'cpus': CPUS,
}
if GENUS:
    prokka_params['genus'] = GENUS
if SPECIES:
    prokka_params['species'] = SPECIES
if STRAIN:
    prokka_params['strain'] = STRAIN
# 运行 Prokka
try:
    print(f"\n{'='*60}")
    print('步骤 1: Prokka 基因注释')
    print(f"{'='*60}")
    prokka_dir = pipeline.run_prokka(
        fna_file=input_fna,
        prefix=OUTPUT_PREFIX,
        kingdom=KINGDOM,
        **prokka_params
    )
    # 统计 Prokka 结果
    prokka_faa = prokka_dir / f"{OUTPUT_PREFIX}.faa"
    if prokka_faa.exists():
        all_proteins = list(SeqIO.parse(prokka_faa, 'fasta'))
        # 过滤序列
        filtered_proteins = [
            seq for seq in all_proteins
            if MIN_SEQ_LENGTH <= len(seq.seq) <= MAX_SEQ_LENGTH
        ]
        print(f"\n{'='*60}")
        print('Prokka 注释完成！')
        print(f"{'='*60}")
        print(f"总蛋白质数: {len(all_proteins)}")
        print(f"符合长度要求的蛋白质: {len(filtered_proteins)} (长度: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH})")
        print(f"不符合要求的: {len(all_proteins) - len(filtered_proteins)}")
        # 显示长度分布
        lengths = [len(seq.seq) for seq in all_proteins]
        print(f"\n序列长度统计:")
        print(f"  最短: {min(lengths)} aa")
        print(f"  最长: {max(lengths)} aa")
        print(f"  平均: {sum(lengths)/len(lengths):.1f} aa")
    print(f"\n✓ Prokka 结果保存在: {prokka_dir}")
except Exception as e:
    print(f"\n{'='*60}")
    print('Prokka 运行失败')
    print(f"{'='*60}")
    print(f'错误: {e}')
    import traceback
    traceback.print_exc()
    raise


## 10. 选择要预测结构的序列

⚠️ **重要**：如果 Prokka 发现的蛋白质数量很多，ESM3 预测可能需要很长时间。

你可以选择：
- 预测全部序列
- 只预测前 N 个序列
- 按长度排序后选择

**建议**：
- Colab 免费版有时间限制，建议一次预测不超过 50-100 个序列
- 如果序列很多，可以分批运行

In [ ]:
# 配置要预测的序列数量
PREDICT_MODE = "first_n"  # 选项: "all" (全部), "first_n" (前N个), "random" (随机N个)
PREDICT_COUNT = 50  # 如果选择 "first_n" 或 "random"，指定数量

# 读取 Prokka 结果
prokka_faa = prokka_dir / f"{OUTPUT_PREFIX}.faa"
all_proteins = list(SeqIO.parse(prokka_faa, "fasta"))

# 过滤长度
filtered_proteins = [
    seq for seq in all_proteins
    if MIN_SEQ_LENGTH <= len(seq.seq) <= MAX_SEQ_LENGTH
]

# 选择要预测的序列
if PREDICT_MODE == "all":
    selected_proteins = filtered_proteins
    print(f"选择模式: 预测全部序列")
elif PREDICT_MODE == "first_n":
    selected_proteins = filtered_proteins[:PREDICT_COUNT]
    print(f"选择模式: 预测前 {PREDICT_COUNT} 个序列")
elif PREDICT_MODE == "random":
    import random
    selected_proteins = random.sample(filtered_proteins, min(PREDICT_COUNT, len(filtered_proteins)))
    print(f"选择模式: 随机选择 {len(selected_proteins)} 个序列")
else:
    selected_proteins = filtered_proteins
    print(f"未知模式，使用全部序列")

print(f"\n{'='*60}")
print("序列选择结果")
print(f"{'='*60}")
print(f"总序列数: {len(all_proteins)}")
print(f"符合长度要求: {len(filtered_proteins)}")
print(f"将要预测: {len(selected_proteins)}")

if len(selected_proteins) > 0:
    # 估算时间
    avg_time_per_seq = 30  # 秒（粗略估计）
    estimated_minutes = (len(selected_proteins) * avg_time_per_seq) / 60
    print(f"\n预计耗时: 约 {estimated_minutes:.1f} 分钟")

    if estimated_minutes > 60:
        print(f"⚠️ 预计时间较长，建议减少序列数量或分批运行")
else:
    print("\n⚠️ 没有符合条件的序列，请调整参数")

# 显示前几个将要预测的序列
if len(selected_proteins) > 0:
    print(f"\n前 5 个将要预测的序列:")
    for i, seq in enumerate(selected_proteins[:5], 1):
        print(f"  {i}. {seq.id[:50]} (长度: {len(seq.seq)} aa)")

## 11. 运行 ESM3 结构预测

⚠️ **注意**：这一步可能需要较长时间，请确保：
1. 已启用 GPU
2. 有足够的时间（建议序列数 < 100）
3. 可以随时停止并保存已完成的结果

In [ ]:
# 运行 ESM3 结构预测
try:
    print(f"\n{'='*60}")
    print("步骤 2: ESM3 结构预测")
    print(f"{'='*60}")
    print(f"将预测 {len(selected_proteins)} 个蛋白质结构\n")

    # 加载 ESM3 模型
    if pipeline.model is None:
        pipeline.load_esm3_model()

    # 开始预测
    from esm.sdk.api import ESMProtein, GenerationConfig

    pdb_files = []
    success_count = 0
    error_count = 0

    # 逐条预测
    for rec in tqdm(selected_proteins, desc="预测结构"):
        try:
            seq = str(rec.seq)
            # 清理 ID 以用作文件名
            name = rec.id.replace('|', '_').replace('/', '_').replace('\\', '_')[:100]
            pdb_file = pipeline.pdb_dir / f"{name}.pdb"

            # 跳过已存在的文件
            if pdb_file.exists():
                pdb_files.append(pdb_file)
                success_count += 1
                continue

            # 使用 ESM3 生成结构
            protein = ESMProtein(sequence=seq)
            protein = pipeline.model.generate(
                protein,
                GenerationConfig(track='structure', num_steps=NUM_STEPS)
            )

            # 保存为 PDB
            protein.to_pdb(str(pdb_file))
            pdb_files.append(pdb_file)
            success_count += 1

        except Exception as e:
            print(f"\n预测失败 {rec.id}: {e}")
            error_count += 1
            continue

    print(f"\n{'='*60}")
    print("结构预测完成！")
    print(f"{'='*60}")
    print(f"成功: {success_count}")
    print(f"失败: {error_count}")
    print(f"PDB 文件保存在: {pipeline.pdb_dir}")

except Exception as e:
    print(f"\n{'='*60}")
    print("ESM3 预测失败")
    print(f"{'='*60}")
    print(f"错误: {e}")
    import traceback
    traceback.print_exc()

    # 即使失败，也显示已完成的数量
    completed = list(pipeline.pdb_dir.glob("*.pdb"))
    if completed:
        print(f"\n已完成 {len(completed)} 个结构预测")
        print(f"文件位置: {pipeline.pdb_dir}")

## 12. 准备 DALI 文件

将预测的结构准备为 DALI 格式

In [ ]:
try:
    # 获取所有已生成的 PDB 文件
    pdb_files = list(pipeline.pdb_dir.glob("*.pdb"))

    if len(pdb_files) == 0:
        print("⚠️ 没有找到 PDB 文件，请先运行 ESM3 预测")
    else:
        print(f"\n找到 {len(pdb_files)} 个 PDB 文件")

        # 准备 DALI 文件
        dali_dir = pipeline.prepare_for_dali(pdb_files)

        print(f"\n{'='*60}")
        print("DALI 文件准备完成！")
        print(f"{'='*60}")
        print(f"\nDALI 文件位置: {dali_dir}")
        print(f"文件数量: {len(pdb_files)}")

        # 列出关键文件
        print(f"\n关键文件:")
        print(f"  - pdb*.ent: DALI 格式的结构文件 ({len(list(dali_dir.glob('*.ent')))} 个)")
        print(f"  - pdb_id_mapping.tsv: 文件名映射表")
        print(f"  - pdb_list.txt: 文件列表")
        print(f"  - README.txt: 使用说明")

except Exception as e:
    print(f"\n{'='*60}")
    print("DALI 文件准备失败")
    print(f"{'='*60}")
    print(f"错误: {e}")
    import traceback
    traceback.print_exc()

## 13. 查看结果摘要

In [ ]:
# 统计结果
print("\n" + "="*60)
print("结果摘要")
print("="*60)

# Prokka 结果
prokka_faa = pipeline.prokka_dir / OUTPUT_PREFIX / f"{OUTPUT_PREFIX}.faa"
if prokka_faa.exists():
    prokka_proteins = list(SeqIO.parse(prokka_faa, "fasta"))
    print(f"\nProkka 注释结果:")
    print(f"  蛋白质数量: {len(prokka_proteins)}")

# ESM3 结果
pdb_files = list(pipeline.pdb_dir.glob("*.pdb"))
print(f"\nESM3 结构预测:")
print(f"  PDB 文件数: {len(pdb_files)}")

# DALI 文件
dali_files = list(pipeline.dali_dir.glob("*.ent"))  # 修正：应该检查 .ent 文件
print(f"\nDALI 输入文件:")
print(f"  准备就绪的文件: {len(dali_files)}")

print(f"\n" + "="*60)
print("\n下一步:")
print("1. 查看上述统计信息确认结果")
print("2. 在服务器上访问以下目录查看结果:")
print(f"   - Prokka 注释: {pipeline.prokka_dir}")
print(f"   - ESM3 结构: {pipeline.pdb_dir}")
print(f"   - DALI 文件: {pipeline.dali_dir}")
print("3. 使用 DALI 进行结构比对:")
print("   访问 http://ekhidna2.biocenter.helsinki.fi/dali/")
print("   上传 dali_ready/ 目录中的 .ent 文件")
print("4. 使用 pdb_id_mapping.tsv 查找原始蛋白质名称")

## 14. 访问结果文件

所有结果已保存在服务器上，可以直接访问

In [ ]:
print("="*60)
print("结果文件位置")
print("="*60)
print(f"\n工作目录: {WORK_DIR.resolve()}")
print(f"\n详细目录:")
print(f"  📁 Prokka 注释结果:")
print(f"     {pipeline.prokka_dir}")
print(f"\n  📁 ESM3 预测的 PDB 结构:")
print(f"     {pipeline.pdb_dir}")
print(f"\n  📁 DALI 格式文件:")
print(f"     {pipeline.dali_dir}")
print(f"\n重要文件:")
print(f"  - {pipeline.dali_dir}/pdb_id_mapping.tsv")
print(f"    (DALI 文件名与原始蛋白质的映射表)")
print(f"  - {pipeline.dali_dir}/README.txt")
print(f"    (DALI 使用说明)")
print(f"\n提示:")
print("  - 可以在 JupyterLab 文件浏览器中直接访问这些目录")
print("  - 也可以使用终端命令查看文件:")
print(f"    cd {WORK_DIR}")
print(f"    ls -lh dali_ready/")
if IN_JUPYTERHUB:
    print("\n  - 服务器上的文件会持久保存，不会因为会话结束而丢失")

## 15. 可选：单独查看某个 PDB 结构

In [ ]:
# 列出所有生成的 PDB 文件
pdb_files = sorted(pipeline.pdb_dir.glob("*.pdb"))

if pdb_files:
    print(f"共 {len(pdb_files)} 个 PDB 文件:\n")
    for i, pdb in enumerate(pdb_files[:10], 1):  # 只显示前10个
        print(f"{i}. {pdb.name}")
    
    if len(pdb_files) > 10:
        print(f"... 还有 {len(pdb_files) - 10} 个文件")
    
    # 如果想查看某个文件内容，可以运行：
    # print("\n第一个 PDB 文件内容（前20行）:")
    # with open(pdb_files[0]) as f:
    #     for i, line in enumerate(f):
    #         if i >= 20:
    #             break
    #         print(line.rstrip())
else:
    print("未找到 PDB 文件")

## 附录：故障排查

### 常见问题

1. **Prokka 安装失败**
   - 确保正确安装了 mamba
   - 尝试重新运行安装单元格

2. **ESM3 显存不足**
   - 减小 `MAX_SEQ_LENGTH` 参数
   - 使用更小的模型或 CPU 模式

3. **Prokka 运行时间过长**
   - 正常情况，取决于输入文件大小
   - 可以增加 `CPUS` 参数

4. **找不到蛋白质序列**
   - 检查输入的 FNA 文件格式是否正确
   - 确认 Prokka 成功运行

### 性能优化

- 使用 GPU 运行时可大幅加速 ESM3
- 调整 `NUM_STEPS` 平衡速度和质量
- 对于大量序列，考虑批量处理